In [3]:

import requests
import pandas as pd
import json
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('YOUTUBE_API_KEY')

if not api_key:
    raise ValueError(' API key not found! Check your .env file.')

print('API Key loaded:', api_key[:10], '...')

API Key loaded: AIzaSyA6Q7 ...


In [4]:
def get_channel_id(channel_name):
    url = 'https://www.googleapis.com/youtube/v3/search'
    params = {
        'part': 'snippet',
        'q': channel_name,
        'type': 'channel',
        'maxResults': 1,
        'key': api_key
    }
    response = requests.get(url, params=params)
    data = response.json()
    channel_id = data['items'][0]['id']['channelId']
    print(f' Channel ID for "{channel_name}": {channel_id}')
    return channel_id

channel_id = get_channel_id('freeCodeCamp')

 Channel ID for "freeCodeCamp": UC8butISFwT-Wl7EV0hUK0BQ


In [5]:
def get_everything(channel_id):
    url = 'https://www.googleapis.com/youtube/v3/channels'
    params = {
        'part': 'snippet,statistics,brandingSettings,contentDetails,topicDetails,status',
        'id': channel_id,
        'key': api_key
    }
    response = requests.get(url, params=params)
    data = response.json()
    channel = data['items'][0]

    everything = {
        # Basic Info
        'channel_id':           channel['id'],
        'channel_name':         channel['snippet']['title'],
        'description':          channel['snippet']['description'],
        'custom_url':           channel['snippet'].get('customUrl', 'N/A'),
        'created_at':           channel['snippet']['publishedAt'][:10],
        'country':              channel['snippet'].get('country', 'N/A'),
        'default_language':     channel['snippet'].get('defaultLanguage', 'N/A'),
        'thumbnail':            channel['snippet']['thumbnails']['high']['url'],
        # Stats
        'subscribers':          channel['statistics'].get('subscriberCount', 'hidden'),
        'total_views':          channel['statistics'].get('viewCount', 0),
        'total_videos':         channel['statistics'].get('videoCount', 0),
        'hidden_subscribers':   channel['statistics'].get('hiddenSubscriberCount', False),
        # Branding
        'channel_keywords':     channel['brandingSettings'].get('channel', {}).get('keywords', 'N/A'),
        'unsubscribed_trailer': channel['brandingSettings'].get('channel', {}).get('unsubscribedTrailer', 'N/A'),
        # Status
        'privacy_status':       channel['status'].get('privacyStatus', 'N/A'),
        'is_linked':            channel['status'].get('isLinked', False),
        'long_upload_status':   channel['status'].get('longUploadsStatus', 'N/A'),
        'made_for_kids':        channel['status'].get('madeForKids', False),
        # Topics
        'topic_categories':     channel.get('topicDetails', {}).get('topicCategories', []),
        # Uploads Playlist
        'uploads_playlist_id':  channel['contentDetails']['relatedPlaylists'].get('uploads', 'N/A'),
    }
    return everything

full_info = get_everything(channel_id)
for key, value in full_info.items():
    print(f'{key:30} → {value}')

channel_id                     → UC8butISFwT-Wl7EV0hUK0BQ
channel_name                   → freeCodeCamp.org
description                    → Learn math, programming, and computer science for free. A 501(c)(3) tax-exempt charity. We also run a free learning interactive platform at freecodecamp.org
custom_url                     → @freecodecamp
created_at                     → 2014-12-16
country                        → US
default_language               → N/A
thumbnail                      → https://yt3.ggpht.com/ytc/AIdro_lGRc-05M2OoE1ejQdxeFhyP7OkJg9h4Y-7CK_5je3QqFI=s800-c-k-c0x00ffffff-no-rj
subscribers                    → 11600000
total_views                    → 966813286
total_videos                   → 2223
hidden_subscribers             → False
channel_keywords               → "coding bootcamp" "learn to code" "software engineer" nonprofits "full stack" "front end" developer programmer javascript python "web development" technology math coding css html "web design" "data science

In [6]:
def get_all_video_ids(channel_id):
    url = 'https://www.googleapis.com/youtube/v3/search'
    video_ids = []
    next_page_token = None

    while True:
        params = {
            'part': 'id',
            'channelId': channel_id,
            'maxResults': 50,
            'type': 'video',
            'key': api_key,
            'pageToken': next_page_token
        }
        response = requests.get(url, params=params)
        data = response.json()

        for item in data['items']:
            video_ids.append(item['id']['videoId'])

        next_page_token = data.get('nextPageToken')
        if not next_page_token:
            break

    print(f'Total video IDs collected: {len(video_ids)}')
    return video_ids

video_ids = get_all_video_ids(channel_id)

Total video IDs collected: 12


In [7]:
df = pd.DataFrame(video_ids)

In [8]:
# ─────────────────────────────────────────
# CELL 5 — Get Full Stats for Every Video
# ─────────────────────────────────────────
def get_video_details(video_ids):
    url = 'https://www.googleapis.com/youtube/v3/videos'
    all_videos = []

    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        params = {
            'part': 'snippet,statistics,contentDetails',
            'id': ','.join(batch),
            'key': api_key
        }
        response = requests.get(url, params=params)
        data = response.json()

        for video in data['items']:
            all_videos.append({
                'video_id':     video['id'],
                'title':        video['snippet']['title'],
                'published_at': video['snippet']['publishedAt'][:10],
                'description':  video['snippet']['description'][:200],
                'tags':         ', '.join(video['snippet'].get('tags', [])),
                'duration':     video['contentDetails']['duration'],
                'views':        video['statistics'].get('viewCount', 0),
                'likes':        video['statistics'].get('likeCount', 0),
                'comments':     video['statistics'].get('commentCount', 0),
                'thumbnail':    video['snippet']['thumbnails']['high']['url'],
                'video_url':    f'https://www.youtube.com/watch?v={video["id"]}'
            })

    print(f' Total videos fetched: {len(all_videos)}')
    return all_videos

videos_data = get_video_details(video_ids)

 Total videos fetched: 12


In [9]:
# ─────────────────────────────────────────
# CELL 6 — Videos DataFrame
# ─────────────────────────────────────────
df = pd.DataFrame(videos_data)

df['views']    = pd.to_numeric(df['views'])
df['likes']    = pd.to_numeric(df['likes'])
df['comments'] = pd.to_numeric(df['comments'])

df = df.sort_values('views', ascending=False).reset_index(drop=True)

print(f' Videos DataFrame shape: {df.shape}')
df.head(10)

 Videos DataFrame shape: (12, 11)


,video_id,title,published_at,description,tags,duration,views,likes,comments,thumbnail,video_url
0,bMknfKXIFA8,React Course - Beginner's Tutorial for React J...,2022-01-10,🎉 Watch the updated version of this course: ht...,,PT11H55M28S,4252230,74171,2651,https://i.ytimg.com/vi/bMknfKXIFA8/hqdefault.jpg,https://www.youtube.com/watch?v=bMknfKXIFA8
1,6ERdu4k62wI,Use PHP to Create an MVC Framework - Full Course,2020-10-22,Learn how to use PHP to build an MVC framework...,"php, php mvc, php mvc framework, php custom ro...",PT6H3M47S,274663,6791,350,https://i.ytimg.com/vi/6ERdu4k62wI/hqdefault.jpg,https://www.youtube.com/watch?v=6ERdu4k62wI
2,zfvxp7PgQ6c,Python and Pygame Tutorial - Build Tetris! Ful...,2018-12-05,Learn how to code Tetris in Python with Pygame...,"python game tutorial, pygame, python, pygame t...",PT1H40M45S,233834,3182,153,https://i.ytimg.com/vi/zfvxp7PgQ6c/hqdefault.jpg,https://www.youtube.com/watch?v=zfvxp7PgQ6c
3,LrZNeyK1xU8,Sexy Typography: CSS Tutorial (Day 2 of CSS3 i...,2018-09-12,Use CSS3 to create visually attractive typogra...,"css, css tutorial, css course, css tutorial fo...",PT11M55S,49387,877,37,https://i.ytimg.com/vi/LrZNeyK1xU8/hqdefault.jpg,https://www.youtube.com/watch?v=LrZNeyK1xU8
4,4m9j6hlbf4g,"IT Fundamentals Course – Hardware, Cloud, DevO...",2026-04-28,This course provides a solid foundation to sta...,,PT13H2M11S,33791,2553,76,https://i.ytimg.com/vi/4m9j6hlbf4g/hqdefault.jpg,https://www.youtube.com/watch?v=4m9j6hlbf4g
5,ozWrlHQGuvI,3D Web Development with Blender and Three.js –...,2026-04-22,Take your creative web development to the next...,,PT5H24M28S,32243,1523,43,https://i.ytimg.com/vi/ozWrlHQGuvI/hqdefault.jpg,https://www.youtube.com/watch?v=ozWrlHQGuvI
6,xapvhkhlPNI,Build a Shopping List for the Command Line - P...,2020-08-25,Learn the basics of Python live from Sam Focht...,,PT1H6M31S,28874,715,20,https://i.ytimg.com/vi/xapvhkhlPNI/hqdefault.jpg,https://www.youtube.com/watch?v=xapvhkhlPNI
7,XKOR4h3CrwE,Gemini CLI Essentials – Full Course,2026-04-24,Learn how to use the Gemini CLI for agentic co...,,PT3H49M40S,24480,696,24,https://i.ytimg.com/vi/XKOR4h3CrwE/hqdefault.jpg,https://www.youtube.com/watch?v=XKOR4h3CrwE
8,wApaJjvNZFs,Stanford's Elite Student Hackathon – Full Docu...,2026-04-29,This documentary will introduce you to the hig...,,PT1H42M23S,10726,622,32,https://i.ytimg.com/vi/wApaJjvNZFs/hqdefault.jpg,https://www.youtube.com/watch?v=wApaJjvNZFs
9,38-3cShsQBc,We all know AI can be...divisive. Justin lays ...,2026-04-27,We all know AI can be...divisive. Justin lays ...,,PT58S,7992,77,1,https://i.ytimg.com/vi/38-3cShsQBc/hqdefault.jpg,https://www.youtube.com/watch?v=38-3cShsQBc


In [10]:

def get_all_playlists(channel_id):
    url = 'https://www.googleapis.com/youtube/v3/playlists'
    playlists = []
    next_page_token = None

    while True:
        params = {
            'part': 'snippet,contentDetails',
            'channelId': channel_id,
            'maxResults': 50,
            'key': api_key,
            'pageToken': next_page_token
        }
        response = requests.get(url, params=params)
        data = response.json()

        for item in data['items']:
            playlists.append({
                'playlist_id':   item['id'],
                'title':         item['snippet']['title'],
                'description':   item['snippet']['description'][:150],
                'published_at':  item['snippet']['publishedAt'][:10],
                'total_videos':  item['contentDetails']['itemCount'],
                'playlist_url':  f'https://www.youtube.com/playlist?list={item["id"]}'
            })

        next_page_token = data.get('nextPageToken')
        if not next_page_token:
            break

    print(f' Total playlists found: {len(playlists)}')
    return playlists

playlists_data = get_all_playlists(channel_id)

 Total playlists found: 74


In [11]:
# ─────────────────────────────────────────
# CELL 8 — Playlists DataFrame
# ─────────────────────────────────────────
df_playlists = pd.DataFrame(playlists_data)
df_playlists = df_playlists.sort_values('total_videos', ascending=False).reset_index(drop=True)

print(f' Playlists DataFrame shape: {df_playlists.shape}')
df_playlists.head(10)

 Playlists DataFrame shape: (74, 6)


,playlist_id,title,description,published_at,total_videos,playlist_url
0,PLWKjhJtqVAbknyJ7hSrf1WKh_Xnv9RL1r,Live Coding with Jesse,,2017-05-05,226,https://www.youtube.com/playlist?list=PLWKjhJt...
1,PLWKjhJtqVAbmDGFE_pZ-PDJ1GWe3KtT-M,Tutorials,,2017-09-19,174,https://www.youtube.com/playlist?list=PLWKjhJt...
2,PLWKjhJtqVAbl9yptoxdSJDDoTVdcysyPo,Talks,,2018-02-07,115,https://www.youtube.com/playlist?list=PLWKjhJt...
3,PLWKjhJtqVAbm04DK8TSUCRheRjW2P9TR7,the freeCodeCamp.org podcast,"Raw, unedited interviews with developers, done...",2024-01-25,109,https://www.youtube.com/playlist?list=PLWKjhJt...
4,PLWKjhJtqVAbkOoQw0jDWn-pzN3nFVjhbH,React Project 5 - Live Coding with Jesse,"In this project we use React, NextJS, Apollo, ...",2018-03-29,103,https://www.youtube.com/playlist?list=PLWKjhJt...
5,PLWKjhJtqVAbmoiNlqLJg1gxEjEuKHHcn_,Beau teaches JavaScript,,2017-02-08,92,https://www.youtube.com/playlist?list=PLWKjhJt...
6,PLWKjhJtqVAbnupwRFOq9zGOWjdvPRtCmO,Full Courses in One Video,,2018-08-02,52,https://www.youtube.com/playlist?list=PLWKjhJt...
7,PLWKjhJtqVAbmfoj2Th9fvxhHIeqFO7wOy,Computer Science and Software Engineering Theo...,,2015-08-03,43,https://www.youtube.com/playlist?list=PLWKjhJt...
8,PLWKjhJtqVAbk2qRZtWSzCIN38JC_NdhW5,JavaScript Basics Course,This is a complete JavaScript course!,2017-02-27,36,https://www.youtube.com/playlist?list=PLWKjhJt...
9,PLWKjhJtqVAbndUuYBE5sVViMIvyzp_dB1,Maths for Programmers,,2016-10-04,35,https://www.youtube.com/playlist?list=PLWKjhJt...


In [12]:
# ─────────────────────────────────────────
# CELL 9 — Combine Everything into One Variable
# ─────────────────────────────────────────
full_channel_data = {
    'channel_info': full_info,
    'videos':       df.to_dict(orient='records'),
    'playlists':    df_playlists.to_dict(orient='records'),
    'summary': {
        'total_videos':            len(df),
        'total_playlists':         len(df_playlists),
        'most_viewed_video':       df.iloc[0]['title'],
        'most_liked_video':        df.loc[df['likes'].idxmax()]['title'],
        'most_commented_video':    df.loc[df['comments'].idxmax()]['title'],
        'avg_views':               int(df['views'].mean()),
        'avg_likes':               int(df['likes'].mean()),
        'avg_comments':            int(df['comments'].mean()),
        'total_views_all_videos':  int(df['views'].sum()),
    }
}

print(' Data stored in full_channel_data')
print(f'Keys: {list(full_channel_data.keys())}')
print(f'\n--- SUMMARY ---')
for key, value in full_channel_data['summary'].items():
    print(f'{key:30} → {value}')

 Data stored in full_channel_data
Keys: ['channel_info', 'videos', 'playlists', 'summary']

--- SUMMARY ---
total_videos                   → 12
total_playlists                → 74
most_viewed_video              → React Course - Beginner's Tutorial for React JavaScript Library [2022]
most_liked_video               → React Course - Beginner's Tutorial for React JavaScript Library [2022]
most_commented_video           → React Course - Beginner's Tutorial for React JavaScript Library [2022]
avg_views                      → 412914
avg_likes                      → 7606
avg_comments                   → 282
total_views_all_videos         → 4954977


In [13]:
# ─────────────────────────────────────────
# CELL 10 — Save to JSON & CSV
# ─────────────────────────────────────────
os.makedirs('data', exist_ok=True)

# Save full data as JSON
with open('data/full_channel_data.json', 'w', encoding='utf-8') as f:
    json.dump(full_channel_data, f, ensure_ascii=False, indent=2)

# Save videos as CSV
df.to_csv('data/videos.csv', index=False)

# Save playlists as CSV
df_playlists.to_csv('data/playlists.csv', index=False)

# Save channel info as JSON
with open('data/channel_info.json', 'w', encoding='utf-8') as f:
    json.dump(full_info, f, ensure_ascii=False, indent=2)

print(' All data saved to /data folder:')
print('   → data/full_channel_data.json')
print('   → data/videos.csv')
print('   → data/playlists.csv')
print('   → data/channel_info.json')

 All data saved to /data folder:
   → data/full_channel_data.json
   → data/videos.csv
   → data/playlists.csv
   → data/channel_info.json
